In [3]:
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import os
import json
from dotenv import load_dotenv
x = load_dotenv()

from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
client = TypeSafeClient()

model_dump = TypeSafeClient().models.list().model_dump()

In [ ]:
def create_metadata(instructions, criteria, task, criteria_dict, instructions_dict, model_dump):
    meta = {}
    meta['Date'] = datetime.now().strftime('%Y/%m/%d %H:%M')
    meta['Task'] = task

    meta['instructions_id'] = instructions
    meta['instructions'] = instructions_dict[instructions]
    meta['criteria_id'] = criteria
    meta['criteria'] = criteria_dict[criteria]

    meta['model'] = 'jev-latest'
    meta['model_metadata'] = next((m for m in model_dump['models'] if m['name'] == meta['model']), None)

    path = f"results/Task{task}_ins-{instructions}_cri-{criteria}.json"
    i = 0

    while os.path.exists(path):
        i += 1
        path = f"results/Task{task}_ins-{instructions}_cri-{criteria}"
        path = f"{path}_{i}.json"

    with open(f'{path}', 'x', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {path}')

    return path

def ask_jev(metadata, state):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)

    instructions = meta['instructions']
    criteria = meta['criteria']

    # print(f"""
    #     instructions = {instructions}\n
    #     criteria = {criteria} \n
    #     state = {state}
    # """
    # )

    starttime = datetime.now()
    response = client.system_one(
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time

In [16]:
criteria_dict = {
    'basic': {
        "contains": "",
        "within": "",
        "touches": "",
        "crosses": "",
        "disjoint": "",
        "overlaps": "",
        "equals": "",
    },
    'explained': {}
}

In [17]:
instructions_dict = {
    'basic': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM.",
    'think': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM. Think step by step.", # Thinking step by step is contradictory to system one intelligence
}

## Task 3 — Conversion of vernacular relation descriptions

Reproduction of the experiment done by Ji et al (2025) using Jev by TypesafeAI instead of GPT-4. The same data and evaluation methods are used. Prompts are adapted to the Jev model and different prompting strategies (instructions and criteria) are tested for each stage.

Stages:  
3-1 No Context  
3-2 Geometry types as context  
3-3 Place types as context

I do not fully understand why only so few samples are taken into the final results in the paper. It is explained, but I don't get it. Should I do all samples?


Ji, Y., Gao, S., Nie, Y., Majić, I., & Janowicz, K. (2025). Foundation models for geospatial reasoning: Assessing the capabilities of large language models in understanding geometries and topological spatial relations. International Journal of Geographical Information Science, 39(9), 1866–1903. https://doi.org/10.1080/13658816.2025.2511227

### Task 3-1: No context

I am confused what the correct answers are supposed to be with this task. One vernacular relation can mean different predicates without context. How is the model supposed to know?

In [29]:
description_list={
    "is in": "within",
    "is located in": "within",
    "is bordered by": "touches",
    "is the county seat of": "within",
    "is home to": "contains",
    "borders": "touches",
    "is surrounded by": "touches",
    "is suburb of": "",
    "has part of the population in": "overlaps",
    "is adjacent to": "touches",
    "is between C and": "touches",
    "share border with": "share a border with",
    "is part of": "",
    "extend into": "overlaps",
    "is the location of": "contains",
    "is an enclave of": "touches",
    "is within": "touches",
    "is halfway between C and": "disjoint",
    "is neighboring": "touches",
    "connect C and": "crosses",
    "is near": "",
    "is mostly in": "",
    "is partly in": "overlaps",
    "is bounded by": "touches",
    "is along": "",
    "surrounds": "",
    "is on": "crosses",
    "includes": "",
    "is midway between C and": "disjoint",
    "on the shore of": "",
    "is situated on": ""
}

In [30]:
len(description_list.keys())

31

In [28]:
path = create_metadata('basic', 'basic', '3-1', criteria_dict, instructions_dict, model_dump)
columns = ['vernacular', 'predicate', 'choice', 'probabilities', 'confidence', 'correct', 'tokens', 'time']
results = []

for vernacular in description_list.keys():
    row = {}
    state = f"Place A {vernacular} Place B"
    response, time = ask_jev(path, state)

    row['vernacular'] = vernacular
    row['predicate'] = description_list[vernacular]
    row['choice'] = response.answers['Task'].choice
    row['probabilities'] = str(response.answers['Task'].probabilities)
    row['confidence'] = response.answers['Task'].confidence
    row['correct'] = int(row['choice']==row['predicate'])
    row['tokens'] = str(response.usage)
    row['time'] = time
    
    results.append(row)

Saved results/Task3-1_ins-basic_cri-basic.json_5


In [21]:
results

[{'vernacular': 'is in',
  'predicate': 'within',
  'choice': 'within',
  'probabilities': "{'contains': 0.05, 'within': 0.95, 'touches': 0.0, 'crosses': 0.0, 'disjoint': 0.0, 'overlaps': 0.0, 'equals': 0.0}",
  'confidence': 0.94,
  'tokens': 'input_tokens=352 output_tokens=73',
  'time': '0:00:00'}]

### Stage 2 — Geometry types as context

In [ ]:
description_list=[['connect C and','2','crosses','geometry_type','LineString/MultiPolygon'],
['extend into','2','overlaps','geometry_type','Polygon/Polygon'],
['is bordered by','2','touches','geometry_type','Polygon/MultiPolygon'],
['is in','2','within','geometry_type','Polygon/MultiPolygon'],
['is neighboring','2','touches','geometry_type','Polygon/Polygon'],
['is on','2','crosses','geometry_type','Polygon/LineString'],
['is surrounded by','2','touches','geometry_type','Polygon/MultiPolygon'],
['is the county seat of','2','within','geometry_type','Polygon/Polygon']]

### Stage 3 — Place types as context

In [ ]:
description_list=[
['borders','1','touches','place_type','city/city'],
['borders','1','touches','place_type','city/municipality'],
['connect C and','1','crosses','place_type','industrial/city'],
['extend into','1','overlaps','place_type','city/county'],
['is adjacent to','1','touches','place_type','city/municipality'],
['is between C and','1','touches','place_type','town/town'],
['is bordered by','1','touches','place_type','town/city'],
['is bounded by','1','touches','place_type','city/city'],
['is home to','1','contains','place_type','city/amenity'],
['is in','1','within','place_type','village/county'],
['is located in','1','within','place_type','city/state'],
['is located in','1','within','place_type','town/county'],
['is located in','1','within','place_type','village/county'],
['is partly in','1','overlaps','place_type','city/county'],
['is surrounded by','1','touches','place_type','city/city'],
['is surrounded by','1','touches','place_type','town/city'],
['is within','1','touches','place_type','city/municipality']]